## Assembling the Transformer Model

Welcome to Bringing Transformers to Life: Training & Inference! Great work in getting to this point. Over the past two courses, you've journeyed from traditional sequence models to mastering the intricate components of the Transformer architecture. In our first course, you explored RNNs, LSTMs, and discovered how attention mechanisms revolutionized sequence modeling. In our second course, you meticulously deconstructed every piece of the Transformer, from multi-head attention and positional encodings to complete encoder and decoder layers.

Now, we enter an exciting new phase where theory meets practice. This course transforms your architectural knowledge into working, trainable models that can tackle real-world tasks. As we begin this first lesson of our third course, we'll take the crucial step of assembling all those carefully crafted components into a unified, fully functional Transformer model. By the end of this lesson, you'll have built a complete implementation that bridges the gap between isolated components and production-ready architectures.
Building the Transformer Foundation

Let's begin constructing our complete Transformer model by establishing its foundational structure. Notice how we bring together all the components we've built separately:
Python
Copy to clipboard

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

This initialization reveals the Transformer's modular design philosophy. We maintain separate embeddings for source and target vocabularies because they often represent different languages or domains; imagine translating from English (with its 170,000+ words) to Chinese (with its character-based system). The d_model parameter acts as the universal dimensionality that ensures all components speak the same mathematical language. By setting max_seq_len to 5000, we pre-compute positional encodings for sequences up to this length, striking a balance between memory efficiency and practical flexibility for most real-world applications.
Integrating Encoder and Decoder Stacks

Now, we integrate the encoder and decoder stacks, along with the crucial output projection layer:
Python
Copy to clipboard

        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)
        
        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_parameters()

The beauty of our modular approach shines here: the TransformerEncoder and TransformerDecoder classes we painstakingly built in previous lessons now slot in seamlessly. The output projection layer deserves special attention — it's essentially asking, "Given everything the decoder has learned about the sequence, what's the probability of each word in my target vocabulary being next?" This linear transformation from d_model dimensions to tgt_vocab_size dimensions creates the logits that, after softmax, become our predicted token probabilities.
Implementing Essential Utilities

Any Machine Learning model needs robust initialization and utility functions to handle real-world data complexities:
Python
Copy to clipboard

    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_padding_mask(self, seq, pad_token=0):
        """
        Create a padding mask for variable-length sequences.
        """
        # (B, S) -> (B, 1, 1, S) to broadcast over heads and query positions
        return (seq != pad_token).unsqueeze(1).unsqueeze(2)
    
    def create_causal_mask(self, size):
        """
        Create a causal (look-ahead) mask for decoder self-attention.
        """
        # (S, S) lower-triangular -> (1, 1, S, S) to broadcast over batch and heads
        mask = torch.tril(torch.ones(size, size))
        return mask.unsqueeze(0).unsqueeze(1).bool()

Here:

    Xavier uniform initialization ensures our model starts training from a stable point — without it, random initialization could lead to vanishing or exploding gradients that make training impossible.
    The create_padding_mask function is crucial for handling variable-length sequences: in a batch containing "Hello world" and "Hi," the shorter sequence gets padded, and this mask ensures the model ignores those meaningless pad tokens. It returns a boolean tensor of shape (batch_size, 1, 1, seq_len), which is broadcastable to attention score tensors of shape (batch_size, num_heads, query_len, key_len).
    The create_causal_mask generates a lower triangular matrix that enforces the decoder's autoregressive property, preventing it from "cheating" by looking at future tokens during training. It returns a boolean tensor of shape (1, 1, size, size), where size is the target length. In practice, the decoder self-attention mask is formed by combining the target padding mask (batch_size, 1, 1, size) with the causal mask (1, 1, size, size) via logical AND, yielding a mask of shape (batch_size, 1, size, size).

Orchestrating the Forward Pass

The forward pass reveals how all components collaborate to transform input sequences into output predictions:
Python
Copy to clipboard

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """Forward pass through complete Transformer"""
        # Embed and encode source
        src_embedded = self.positional_encoding(self.src_embedding(src))
        encoder_output, _ = self.encoder(src_embedded, src_mask)
        
        # Embed and decode target
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
        decoder_output, _ = self.decoder(tgt_embedded, encoder_output, tgt_mask, src_mask)
        
        # Project to vocabulary
        output = self.output_projection(decoder_output)
        return output

This elegantly simple forward pass conceals sophisticated interactions. Source tokens undergo a three-stage transformation: embedding lookup converts integers to vectors, positional encoding adds sequence order information, and the encoder stack builds rich contextual representations. The decoder follows a parallel path but with a crucial difference: it receives the encoder's output as additional context through cross-attention mechanisms. The final projection transforms the decoder's abstract representations back into concrete vocabulary predictions, completing the sequence-to-sequence pipeline.
Setting up the Transformer

Let's validate our implementation through systematic testing that demonstrates the model's capabilities. We'll begin by setting up the model configuration and verifying its basic structure:
Python
Copy to clipboard

def main():    
    # Configuration
    src_vocab_size = 1000
    tgt_vocab_size = 1200
    d_model = 128
    num_heads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    d_ff = 512
    batch_size = 2
    src_seq_len = 10
    tgt_seq_len = 8
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        d_ff=d_ff
    )
    
    # Create sample inputs
    torch.manual_seed(42)
    src_tokens = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")

This configuration uses deliberately modest dimensions to enable quick verification while maintaining architectural authenticity. The different vocabulary sizes (1000 vs. 1200) demonstrate the model's flexibility, essential for tasks like translating between languages with vastly different vocabulary sizes:
text
Copy to clipboard

Model parameters: 1,362,096
Source tokens shape: torch.Size([2, 10])
Target tokens shape: torch.Size([2, 8])

Testing Forward Pass and Gradient Flow

Now, let's execute the forward pass and verify that our model produces the expected outputs with proper gradient flow:
Python
Copy to clipboard

    # Create masks
    src_mask = model.create_padding_mask(src_tokens)
    tgt_mask = model.create_causal_mask(tgt_seq_len)
    
    # Forward pass
    output = model(src_tokens, tgt_tokens, src_mask, tgt_mask)
    
    print(f"Model output shape: {output.shape}")
    print(f"Expected shape: ({batch_size}, {tgt_seq_len}, {tgt_vocab_size})")
    
    # Test gradient flow
    loss = output.mean()
    loss.backward()
    
    print("Gradient verification:")
    print(f"  Source embedding grad: {model.src_embedding.embedding.weight.grad.norm():.6f}")
    print(f"  Output projection grad: {model.output_projection.weight.grad.norm():.6f}")

The gradient flow verification is particularly crucial — it confirms that gradients propagate seamlessly from the output projection layer all the way back to the source embeddings. This end-to-end gradient flow ensures every parameter can learn during training:
text
Copy to clipboard

Model output shape: torch.Size([2, 8, 1200])
Expected shape: (2, 8, 1200)
Gradient verification:
  Source embedding grad: 0.050715
  Output projection grad: 0.268537

With over 1.3 million parameters, our model has substantial capacity while remaining computationally manageable for educational exploration.
Conclusion and Next Steps

You've successfully assembled a complete, production-ready Transformer model that elegantly integrates every component from our architectural journey. This achievement represents more than technical implementation: you now possess a deep understanding of how token embeddings, positional encodings, attention mechanisms, and projection layers collaborate to enable powerful sequence-to-sequence transformations. Your model stands ready to tackle any task, from machine translation to text summarization, from dialogue generation to code completion.

The practice exercises ahead will solidify your understanding through hands-on experimentation with this complete architecture. As we progress through this course, we'll explore how to prepare datasets, implement efficient training loops, and deploy these models for real-world inference, transforming your theoretical mastery into practical expertise that can solve meaningful problems.

## Building Your First Transformer Model

Now that you've learned about the individual components of the Transformer architecture, it's time to bring them all together into a complete model. You'll build the main `Transformer` class that connects all the pieces you've studied so far.

---

### Your Task

Your goal is to complete the provided code in two files:

- In `model.py`, implement the `__init__` constructor by creating all the necessary components (embeddings, encoder, decoder, output projection), and implement the `_init_parameters` method for proper weight initialization.
- In `main.py`, set up the configuration parameters and create a model instance to test that everything works.

The constructor should store the model dimension, create token embeddings for both source and target vocabularies, add positional encoding, build the encoder and decoder stacks, and create the final output projection layer. The parameter initialization method should apply Xavier uniform initialization to all parameters with more than one dimension.

This is a big step forward, you're creating the complete architecture that powers modern language models!

---

### Starter Code

```python

## model.py
import torch
import torch.nn as nn

from transformer.encoder import TransformerEncoder
from transformer.decoder import TransformerDecoder
from transformer.embeddings import TokenEmbedding, PositionalEncoding

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        # TODO: Store d_model as instance variable
        
        # TODO: Create source embedding layer using TokenEmbedding
        
        # TODO: Create target embedding layer using TokenEmbedding
        
        # TODO: Create positional encoding layer using PositionalEncoding
        
        # TODO: Create encoder using TransformerEncoder
        
        # TODO: Create decoder using TransformerDecoder
        
        # TODO: Create output projection layer using nn.Linear
        
        # TODO: Call the parameter initialization method
    
    def _init_parameters(self):
        """Initialize model parameters"""
        # TODO: Loop through all parameters and apply Xavier uniform initialization
        pass


## main.py
import torch

from transformer.model import Transformer

def main():
    """Test Transformer model creation"""
    
    # TODO: Define configuration parameters
    # Set src_vocab_size
    # Set tgt_vocab_size
    # Set d_model
    # Set num_heads 
    # Set num_encoder_layers
    # Set num_decoder_layers
    # Set d_ff
    
    # TODO: Create model using the Transformer class with the configuration parameters
    
    # TODO: Print the total number of model parameters
    # TODO: Print the model's d_model value
    

if __name__ == "__main__":
    main()
```

---

### Solution

The TODOs in `model.py` are in `Transformer.__init__` and `_init_parameters`:

```python
# Store d_model as instance variable
self.d_model = d_model

# Embeddings: separate source and target vocabularies
self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

# Encoder and Decoder
self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)

# Output projection: d_model -> target vocabulary logits
self.output_projection = nn.Linear(d_model, tgt_vocab_size)

# Initialize parameters
self._init_parameters()
```

```python
def _init_parameters(self):
    """Initialize model parameters"""
    for p in self.parameters():
        if p.dim() > 1:  # weight matrices only, not biases
            nn.init.xavier_uniform_(p)
```

- `self.d_model` is stored because the model dimension is the shared size that every component uses.
- The source and target sides get their own `TokenEmbedding`, sized by `src_vocab_size` and `tgt_vocab_size`, since the two languages can have different vocabularies. One `PositionalEncoding(d_model, max_seq_len, dropout)` is shared, because positions work the same way on both sides.
- The encoder and decoder stacks receive `num_encoder_layers` and `num_decoder_layers` respectively, with the same `d_model`, `num_heads`, `d_ff`, `dropout` and `activation`.
- `nn.Linear(d_model, tgt_vocab_size)` turns each decoder vector into one logit per target token.
- `_init_parameters` must be called last in `__init__`, so every layer already exists when the loop runs. `p.dim() > 1` selects the weight matrices and skips the biases and the LayerNorm parameters, which are one-dimensional.
- Without the fix, the model has no layers and `_init_parameters` does nothing because of `pass`. Any later use of `self.src_embedding`, `self.encoder` or `self.output_projection` fails with an `AttributeError`.

The starter code imports the encoder, decoder and embedding classes from CodeSignal's `transformer` package, which does not exist in this notebook. The `model.py` cell below defines all of them inline, using the versions built in the earlier courses, and adds the mask and `forward` methods from the lesson, which are not part of the starter.

The `main.py` starter was not pasted, so its cell is rebuilt from the task text and the lesson. It sets the configuration (vocabularies of 1000 and 1200 tokens, `d_model=128`, 8 heads, 2 encoder and 2 decoder layers, `d_ff=512`), creates the `Transformer`, and checks the forward pass and the gradients. Replace it with your own `main.py` if it differs. In the notebook it relies on `Transformer` from the `model.py` cell, so run that cell first.

## model.py

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """Sinusoidal Positional Encoding"""
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Encoder Layer"""
        super(TransformerEncoderLayer, self).__init__()
        
        # Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
    
    def forward(self, x, mask=None):
        """Forward pass through encoder layer"""
        # Multi-Head Self-Attention + Add & Norm
        attn_output, attention_weights = self.self_attention(x, x, x, mask)
        x = self.add_norm1(x, attn_output)
        
        # Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm2(x, ff_output)
        
        return x, attention_weights

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Encoder Layers"""
        super(TransformerEncoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, mask=None):
        """Forward pass through encoder stack"""
        attention_weights = []
        
        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            attention_weights.append(attn_weights)
        
        return x, attention_weights


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers: one per sub-layer
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)  # residual + layer norm
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        # Initialize empty lists for storing attention weights from each layer
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        # Pass the inputs through each decoder layer in sequence
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)  # x is updated by each layer
            all_self_attention_weights.append(self_attn_weights)  # collect per-layer weights
            all_cross_attention_weights.append(cross_attn_weights)
        
        # Return the final output and all collected attention weights
        return x, (all_self_attention_weights, all_cross_attention_weights)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        # Store d_model as instance variable
        self.d_model = d_model
        
        # Embeddings: separate source and target vocabularies
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)
        
        # Output projection: d_model -> target vocabulary logits
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        # Initialize parameters
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:  # weight matrices only, not biases
                nn.init.xavier_uniform_(p)
    
    # The methods below come from the lesson (not part of the starter code); the quick check needs them.
    def create_padding_mask(self, seq, pad_token=0):
        """Create a padding mask for variable-length sequences."""
        # (B, S) -> (B, 1, 1, S) to broadcast over heads and query positions
        return (seq != pad_token).unsqueeze(1).unsqueeze(2)
    
    def create_causal_mask(self, size):
        """Create a causal (look-ahead) mask for decoder self-attention."""
        # (S, S) lower-triangular -> (1, 1, S, S) to broadcast over batch and heads
        mask = torch.tril(torch.ones(size, size))
        return mask.unsqueeze(0).unsqueeze(1).bool()
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """Forward pass through complete Transformer"""
        # Embed and encode source
        src_embedded = self.positional_encoding(self.src_embedding(src))
        encoder_output, _ = self.encoder(src_embedded, src_mask)
        
        # Embed and decode target
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
        decoder_output, _ = self.decoder(tgt_embedded, encoder_output, tgt_mask, src_mask)
        
        # Project to vocabulary
        output = self.output_projection(decoder_output)
        return output


## main.py

In [ ]:
# main.py (rebuilt from the task text and the lesson, because its starter was not pasted)
# In the notebook, `Transformer` is already defined by the model.py cell above.
# In a separate file you would add: from model import Transformer
import torch

def main():
    # Configuration
    src_vocab_size = 1000
    tgt_vocab_size = 1200
    d_model = 128
    num_heads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    d_ff = 512
    batch_size = 2
    src_seq_len = 10
    tgt_seq_len = 8
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        d_ff=d_ff
    )
    
    # Create sample inputs
    torch.manual_seed(42)
    src_tokens = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")
    
    # Create masks
    src_mask = model.create_padding_mask(src_tokens)
    tgt_mask = model.create_causal_mask(tgt_seq_len)
    
    # Forward pass
    output = model(src_tokens, tgt_tokens, src_mask, tgt_mask)
    
    print(f"Model output shape: {output.shape}")
    print(f"Expected shape: ({batch_size}, {tgt_seq_len}, {tgt_vocab_size})")
    
    # Test gradient flow
    loss = output.mean()
    loss.backward()
    
    print("Gradient verification:")
    print(f"  Source embedding grad: {model.src_embedding.embedding.weight.grad.norm():.6f}")
    print(f"  Output projection grad: {model.output_projection.weight.grad.norm():.6f}")


main()


## Implementing Attention Boundaries

Now that you've set up the Transformer class structure, it's time to implement the masking mechanisms that are crucial for proper attention behavior.

In this exercise, you'll complete the `create_padding_mask` and `create_causal_mask` methods in your `Transformer` class. The padding mask helps the model ignore padding tokens, while the causal mask ensures the decoder can look only at previous tokens, not future ones.

---

### Your Task

- Implement the padding mask method to identify non-padding tokens.
- Create the causal mask using a lower triangular pattern.
- Use both mask methods in the `main` function with your sample inputs.
- Print the mask shapes to verify that they work correctly.

These masks are essential for making your Transformer work properly with real sequence data!

---

### Starter Code

#### model.py

```python
import torch
import torch.nn as nn

from transformer.encoder import TransformerEncoder
from transformer.decoder import TransformerDecoder
from transformer.embeddings import TokenEmbedding, PositionalEncoding

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)
        
        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_padding_mask(self, seq, pad_token=0):
        """Create padding mask"""
        # TODO: Create a mask where padding tokens are marked as False
        # Hint: Compare seq with pad_token and add a dimension
        pass
    
    def create_causal_mask(self, size):
        """Create causal mask for decoder"""
        # TODO: Create a lower triangular mask to prevent looking at future tokens
        # Hint: Use torch.tril to create the triangular pattern
        pass
```

#### main.py

```python
import torch

from transformer.model import Transformer

def main():
    """Test complete Transformer model"""
    
    # Configuration
    src_vocab_size = 1000
    tgt_vocab_size = 1200
    d_model = 128
    num_heads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    d_ff = 512
    batch_size = 2
    src_seq_len = 10
    tgt_seq_len = 8
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        d_ff=d_ff
    )
    
    # Create sample inputs
    torch.manual_seed(42)
    src_tokens = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")
    
    # TODO: Create masks using the model's mask methods
    # Create source padding mask from src_tokens
    # Create target causal mask with tgt_seq_len size
    
    # TODO: Print the mask shapes to verify they were created correctly
    

if __name__ == "__main__":
    main()
```

---

### Solution

Two parts were missing. In `model.py`, the two mask methods:

```python
def create_padding_mask(self, seq, pad_token=0):
    """Create padding mask"""
    # (B, S) -> (B, 1, 1, S): True for real tokens, False for padding
    return (seq != pad_token).unsqueeze(1).unsqueeze(2)

def create_causal_mask(self, size):
    """Create causal mask for decoder"""
    # (S, S) lower triangle -> (1, 1, S, S): position i sees only positions <= i
    mask = torch.tril(torch.ones(size, size))
    return mask.unsqueeze(0).unsqueeze(1).bool()
```

In `main.py`, creating the masks and printing their shapes:

```python
src_mask = model.create_padding_mask(src_tokens)
tgt_mask = model.create_causal_mask(tgt_seq_len)

print(f"Source padding mask shape: {src_mask.shape}")
print(f"Target causal mask shape: {tgt_mask.shape}")
```

- `seq != pad_token` is `True` for real tokens and `False` for padding. The two `unsqueeze` calls turn `(batch, seq_len)` into `(batch, 1, 1, seq_len)`, so the mask broadcasts over the attention heads and the query positions.
- `torch.tril(torch.ones(size, size))` keeps the lower triangle, so position $i$ can attend only to positions $j \le i$. Two `unsqueeze` calls give `(1, 1, size, size)` to broadcast over batch and heads, and `.bool()` makes it a boolean mask like the padding mask.
- In `main.py`, `src_tokens` has shape `[2, 10]` and `tgt_seq_len` is 8, so the printed shapes should be `[2, 1, 1, 10]` for the padding mask and `[1, 1, 8, 8]` for the causal mask.
- Without the fix, both methods return `None` because of `pass`. The mask variables would be `None`, and printing `.shape` on them would raise an `AttributeError`.

The starter code imports the encoder, decoder and embedding classes from CodeSignal's `transformer` package, which does not exist in this notebook. The `model.py` cell below defines all of them inline, using the versions built in the earlier courses. The starter `Transformer` has no `forward` method, so none is added here. The `main.py` cell relies on `Transformer` from the `model.py` cell, so run that cell first.

## model.py

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """Sinusoidal Positional Encoding"""
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Encoder Layer"""
        super(TransformerEncoderLayer, self).__init__()
        
        # Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
    
    def forward(self, x, mask=None):
        """Forward pass through encoder layer"""
        # Multi-Head Self-Attention + Add & Norm
        attn_output, attention_weights = self.self_attention(x, x, x, mask)
        x = self.add_norm1(x, attn_output)
        
        # Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm2(x, ff_output)
        
        return x, attention_weights

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Encoder Layers"""
        super(TransformerEncoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, mask=None):
        """Forward pass through encoder stack"""
        attention_weights = []
        
        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            attention_weights.append(attn_weights)
        
        return x, attention_weights


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers: one per sub-layer
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)  # residual + layer norm
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        # Initialize empty lists for storing attention weights from each layer
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        # Pass the inputs through each decoder layer in sequence
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)  # x is updated by each layer
            all_self_attention_weights.append(self_attn_weights)  # collect per-layer weights
            all_cross_attention_weights.append(cross_attn_weights)
        
        # Return the final output and all collected attention weights
        return x, (all_self_attention_weights, all_cross_attention_weights)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)
        
        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_padding_mask(self, seq, pad_token=0):
        """Create padding mask"""
        # (B, S) -> (B, 1, 1, S): True for real tokens, False for padding
        return (seq != pad_token).unsqueeze(1).unsqueeze(2)
    
    def create_causal_mask(self, size):
        """Create causal mask for decoder"""
        # (S, S) lower triangle -> (1, 1, S, S): position i sees only positions <= i
        mask = torch.tril(torch.ones(size, size))
        return mask.unsqueeze(0).unsqueeze(1).bool()


## main.py

In [ ]:
import torch

# In the notebook, `Transformer` is already defined by the model.py cell above.
# In a separate file you would add: from model import Transformer
from transformer.model import Transformer

def main():
    """Test complete Transformer model"""
    
    # Configuration
    src_vocab_size = 1000
    tgt_vocab_size = 1200
    d_model = 128
    num_heads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    d_ff = 512
    batch_size = 2
    src_seq_len = 10
    tgt_seq_len = 8
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        d_ff=d_ff
    )
    
    # Create sample inputs
    torch.manual_seed(42)
    src_tokens = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")
    
    # Create masks using the model's mask methods
    src_mask = model.create_padding_mask(src_tokens)  # (batch, 1, 1, src_seq_len)
    tgt_mask = model.create_causal_mask(tgt_seq_len)  # (1, 1, tgt_seq_len, tgt_seq_len)

    # Print the mask shapes to verify they were created correctly
    print(f"Source padding mask shape: {src_mask.shape}")
    print(f"Target causal mask shape: {tgt_mask.shape}")
    

if __name__ == "__main__":
    main()


## Your Transformer Comes Alive

Now that you've set up the mask creation methods, it's time to bring your Transformer model to life by implementing the forward pass. This is where all the components you've built come together to process input sequences and generate outputs.

Your main goal is to complete the `forward` method in the `Transformer` class. The method should process the source and target sequences through the full pipeline.

---

### Your Task

- Apply embeddings and positional encoding to both the source and target sequences.
- Pass the source through the encoder to get contextual representations.
- Pass the target through the decoder, using the encoder output for cross-attention.
- Project the decoder output to the vocabulary size to get the final logits.

You'll also need to complete the `main` function to test your implementation. Run the model with sample inputs, verify that the output shape matches expectations, and test gradient flow by computing a simple loss and running backpropagation.

This step transforms your model from a collection of components into a working sequence-to-sequence system, ready for training and inference.

---

### Starter Code

#### model.py

```python
import torch
import torch.nn as nn

from transformer.encoder import TransformerEncoder
from transformer.decoder import TransformerDecoder
from transformer.embeddings import TokenEmbedding, PositionalEncoding

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)
        
        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_padding_mask(self, seq, pad_token=0):
        """Create padding mask"""
        return (seq != pad_token).unsqueeze(1).unsqueeze(2)
    
    def create_causal_mask(self, size):
        """Create causal mask for decoder"""
        mask = torch.tril(torch.ones(size, size))
        return mask.unsqueeze(0).unsqueeze(1).bool()
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """Forward pass through complete Transformer"""
        # TODO: Implement the forward pass
        # 1. Embed and encode the source sequence
        # 2. Embed and decode the target sequence
        # 3. Project to vocabulary size
        pass
```

#### main.py

```python
import torch

from transformer.model import Transformer

def main():
    """Test complete Transformer model"""
    
    # Configuration
    src_vocab_size = 1000
    tgt_vocab_size = 1200
    d_model = 128
    num_heads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    d_ff = 512
    batch_size = 2
    src_seq_len = 10
    tgt_seq_len = 8
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        d_ff=d_ff
    )
    
    # Create sample inputs
    torch.manual_seed(42)
    src_tokens = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")
    
    # Create masks
    src_mask = model.create_padding_mask(src_tokens)
    tgt_mask = model.create_causal_mask(tgt_seq_len)
    
    # TODO: Run forward pass through the model
    # TODO: Print the output shape and verify it matches expected dimensions
    
    # TODO: Test gradient flow by computing loss and running backward pass
    # TODO: Print gradient norms to verify gradients are flowing properly
    

if __name__ == "__main__":
    main()
```

---

### Solution

Two parts were missing. In `model.py`, the `forward` method:

```python
def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    # 1. Embed and encode the source sequence
    src_embedded = self.positional_encoding(self.src_embedding(src))
    encoder_output, _ = self.encoder(src_embedded, src_mask)

    # 2. Embed and decode the target sequence (cross-attention uses src_mask)
    tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
    decoder_output, _ = self.decoder(tgt_embedded, encoder_output, tgt_mask, src_mask)

    # 3. Project to vocabulary size
    return self.output_projection(decoder_output)
```

In `main.py`, the forward pass, shape check and gradient check:

```python
output = model(src_tokens, tgt_tokens, src_mask, tgt_mask)
print(f"Output shape: {output.shape}")
print(f"Expected shape: ({batch_size}, {tgt_seq_len}, {tgt_vocab_size})")

loss = output.sum()
loss.backward()
grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
print(f"Gradient norm (first parameter): {grad_norms[0]:.6f}")
print(f"Gradient norm (last parameter): {grad_norms[-1]:.6f}")
```

- The source goes through `src_embedding`, then `positional_encoding`, then the encoder. The encoder returns `(output, attention_weights)`, so the weights are discarded with `_`.
- The target takes the same path with `tgt_embedding`. The decoder gets `encoder_output` for cross-attention, `tgt_mask` (the causal mask) for self-attention and `src_mask` (the padding mask) for cross-attention, in that order.
- `output_projection` maps `d_model` to `tgt_vocab_size`, so the logits have shape `(batch, tgt_seq_len, tgt_vocab_size)`. Here that is `[2, 8, 1200]`.
- `output.sum()` is only a simple scalar loss to exercise backpropagation. Gradients that are not `None` and have a finite norm show the whole graph (embeddings, encoder, decoder, projection) is connected.
- Without the fix, `forward` returns `None` because of `pass`, so `output.shape` would raise an `AttributeError` and `loss.backward()` could not run.

The starter code imports the encoder, decoder and embedding classes from CodeSignal's `transformer` package, which does not exist in this notebook. The `model.py` cell below defines all of them inline, using the versions built in the earlier courses. The `main.py` cell relies on `Transformer` from the `model.py` cell, so run that cell first. The model is not seeded, so the exact gradient norms differ from run to run.

## model.py

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """Sinusoidal Positional Encoding"""
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Encoder Layer"""
        super(TransformerEncoderLayer, self).__init__()
        
        # Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
    
    def forward(self, x, mask=None):
        """Forward pass through encoder layer"""
        # Multi-Head Self-Attention + Add & Norm
        attn_output, attention_weights = self.self_attention(x, x, x, mask)
        x = self.add_norm1(x, attn_output)
        
        # Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm2(x, ff_output)
        
        return x, attention_weights

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Encoder Layers"""
        super(TransformerEncoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, mask=None):
        """Forward pass through encoder stack"""
        attention_weights = []
        
        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            attention_weights.append(attn_weights)
        
        return x, attention_weights


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers: one per sub-layer
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)  # residual + layer norm
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        # Initialize empty lists for storing attention weights from each layer
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        # Pass the inputs through each decoder layer in sequence
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)  # x is updated by each layer
            all_self_attention_weights.append(self_attn_weights)  # collect per-layer weights
            all_cross_attention_weights.append(cross_attn_weights)
        
        # Return the final output and all collected attention weights
        return x, (all_self_attention_weights, all_cross_attention_weights)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, 
                 max_seq_len=5000, dropout=0.1, activation='relu'):
        """Complete Transformer model for sequence-to-sequence tasks"""
        super(Transformer, self).__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder and Decoder
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, d_ff, dropout, activation)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout, activation)
        
        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize model parameters"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_padding_mask(self, seq, pad_token=0):
        """Create padding mask"""
        return (seq != pad_token).unsqueeze(1).unsqueeze(2)
    
    def create_causal_mask(self, size):
        """Create causal mask for decoder"""
        mask = torch.tril(torch.ones(size, size))
        return mask.unsqueeze(0).unsqueeze(1).bool()
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """Forward pass through complete Transformer"""
        # 1. Embed and encode the source sequence
        src_embedded = self.positional_encoding(self.src_embedding(src))
        encoder_output, _ = self.encoder(src_embedded, src_mask)

        # 2. Embed and decode the target sequence (cross-attention uses src_mask)
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
        decoder_output, _ = self.decoder(tgt_embedded, encoder_output, tgt_mask, src_mask)

        # 3. Project to vocabulary size: (B, T, d_model) -> (B, T, tgt_vocab_size)
        return self.output_projection(decoder_output)


## main.py

In [ ]:
import torch

# In the notebook, `Transformer` is already defined by the model.py cell above.
# In a separate file you would add: from model import Transformer

def main():
    """Test complete Transformer model"""
    
    # Configuration
    src_vocab_size = 1000
    tgt_vocab_size = 1200
    d_model = 128
    num_heads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    d_ff = 512
    batch_size = 2
    src_seq_len = 10
    tgt_seq_len = 8
    
    # Create model
    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        d_ff=d_ff
    )
    
    # Create sample inputs
    torch.manual_seed(42)
    src_tokens = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")
    
    # Create masks
    src_mask = model.create_padding_mask(src_tokens)
    tgt_mask = model.create_causal_mask(tgt_seq_len)
    
    # Run forward pass through the model
    output = model(src_tokens, tgt_tokens, src_mask, tgt_mask)

    # Print the output shape and verify it matches expected dimensions
    print(f"Output shape: {output.shape}")
    print(f"Expected shape: ({batch_size}, {tgt_seq_len}, {tgt_vocab_size})")

    # Test gradient flow by computing loss and running backward pass
    loss = output.sum()
    loss.backward()

    # Print gradient norms to verify gradients are flowing properly
    grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
    print(f"Gradient norm (first parameter): {grad_norms[0]:.6f}")
    print(f"Gradient norm (last parameter): {grad_norms[-1]:.6f}")
    

if __name__ == "__main__":
    main()
